## 5. Algoritmo di Pattern Matching Baseline (Cosine Similarity & 1-NN)

Successivamente alla fase di Exploratory Data Analysis (EDA), il presente modulo implementa il nucleo predittivo del sistema: un classificatore euristico basato su **K-Nearest Neighbors con $K=1$**, storicamente derivato dall'architettura originale in ambiente MQL4 e ora riscritto in Python con un'infrastruttura di calcolo matriciale ottimizzata.

### 5.1 Formulazione Matematica e Stazionarietà Geometrica
Nelle serie storiche finanziarie ad alta frequenza (M1), l'utilizzo dei prezzi assoluti (es. quotazioni dell'Oro a $2400$) introduce una forte dipendenza dalla scala dei prezzi (non-stazionarietà) che compromette il calcolo delle distanze geometriche. Per ovviare a questo limite, il sistema mappa ciascuna candela in uno spazio vettoriale a **4 dimensioni stazionarie**:
1. **Body**: $Close - Open$ (ampiezza e direzione del corpo).
2. **Range**: $High - Low$ (volatilità ed escursione totale).
3. **Upper Shadow**: $High - \max(Open, Close)$ (pressione di vendita / rifiuto dei massimi).
4. **Lower Shadow**: $\min(Open, Close) - Low$ (pressione d'acquisto / rifiuto dei minimi).

Applicando una tecnica di **Flattening**, una finestra temporale (Sliding Window) composta da $W$ candele (default: 10) viene appiattita in un unico vettore unidimensionale di dimensione $W \times 4$ (40 feature geometriche).

### 5.2 Prevenzione del Data Leakage e Validazione Out-of-Sample
Per evitare l'errore logico del *Data Leakage* (sovrapposizione tra dati di training e target predittivo), la pipeline separa rigorosamente:
* **Matrice dei Pattern ($X$):** Le $10$ candele storiche consecutive che formano la forma geometrica di riferimento.
* **Vettore delle Target Label ($y$):** La polarità direzionale della candela immediatamente successiva ed esterna alla finestra ($t+1$), codificata come binaria ($0 =$ Rialzista/Flat, $1 =$ Ribassista).

Il sistema calcola la similarità del coseno tra il pattern corrente e l'intera storia passata, identificando il pattern storico più simile (Nearest Neighbor) tramite la massima similarità del coefficiente angolare, e trasferendo la sua etichetta predittiva al momento presente.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

"""
=========================================================================================
MODULO: Feature Engineering, Sliding Windows & Baseline Cosine Similarity

OBIETTIVO: 
1. Trasformare la serie temporale in finestre scorrevoli geometriche (Matrice X e Vettore y).
2. Eseguire il Pattern Matching vettoriale (Cosine Similarity / 1-NN) per identificare
   il pattern storico più affine e generare una previsione direzionale Out-of-Sample.
=========================================================================================
"""

def create_sliding_windows(df, window_size=10):
    """
    Esegue il flattening delle finestre temporali e calcola le 4 feature geometriche spaziali.
    Isola rigorosamente la Target Label sulla candela successiva per prevenire Data Leakage.
    """
    df_features = df.copy()

    # 1. TRASFORMAZIONE SPAZIALE: Calcolo delle 4 feature intra-candela
    df_features['Body'] = df_features['Close'] - df_features['Open']
    df_features['Range'] = df_features['High'] - df_features['Low']
    df_features['Upper_Shadow'] = df_features['High'] - df_features[['Open', 'Close']].max(axis=1)
    
    df_features['Lower_Shadow'] = df_features[['Open', 'Close']].min(axis=1) - df_features['Low']

    # 2. DEFINIZIONE DELLA LABEL (y): Direzione della candela successiva (0=Rialzista, 1=Ribassista)
    df_features['Target'] = np.where(df_features['Body'] >= 0, 0, 1)

    # 3. INIZIALIZZAZIONE STRUTTURE DATI
    X, y = [], []
    features_array = df_features[['Body', 'Range', 'Upper_Shadow', 'Lower_Shadow']].values
    target_array = df_features['Target'].values

    # 4. SLIDING WINDOW (Scorrimento Temporale)
    for i in range(len(df_features) - window_size):
        window = features_array[i : i + window_size].flatten()
        label = target_array[i + window_size]
        X.append(window)
        y.append(label)

    return np.array(X), np.array(y)

def predict_next_candle(target_pattern, history_X, history_y):
    """
    Calcola la Cosine Similarity tra il pattern di input e l'intero storico,
    restituendo l'indice del match ottimale, lo score geometrico e la previsione.
    """
    target_reshaped = target_pattern.reshape(1, -1)
    
    # Calcolo massimizzato tramite libreria scientifica C-optimized
    similarities = cosine_similarity(target_reshaped, history_X)[0]
    
    best_match_idx = np.argmax(similarities)
    best_similarity_score = similarities[best_match_idx]
    prediction = history_y[best_match_idx]
    
    return best_match_idx, best_similarity_score, prediction

# ==========================================
# ESECUZIONE DELLA PIPELINE DI PREVISIONE
# ==========================================
if __name__ == "__main__":
    # Caricamento dinamico del dataset ML-Ready dalla root del progetto
    file_path = os.path.join(os.getcwd(), 'Data Management', 'ReadyData', 'XAUUSD_ReadyToUse.csv')
    print(f" Caricamento del dataset da: {file_path}")
    df = pd.read_csv(file_path, parse_dates=['Datetime'])
    
    # Esclusione dei macro-gap (weekend) per preservare la serie temporale
    df_clean = df[df['Missing'] == False].copy().reset_index(drop=True)
    
    w_size = 10
    expected_windows = len(df_clean) - w_size
    print(f" Dataset pulito: {len(df_clean)} candele valide.")
    print(f" Generazione stimata di {expected_windows} Sliding Windows in corso...\n")
    
    # Generazione della matrice X e del vettore y
    X, y = create_sliding_windows(df_clean, window_size=w_size)
    print(f" Vettorizzazione completata. Matrice X: {X.shape}, Vettore y: {y.shape}")
    
    # ==========================================
    # TEST DI PREVISIONE OUT-OF-SAMPLE (1-NN)
    # ==========================================
    print("\n Avvio del Pattern Matching (Cosine Similarity 1-NN)...")
    
    # Simulazione operativa: isoliamo l'ultima finestra disponibile come test corrente
    history_X = X[:-1]
    history_y = y[:-1]
    
    current_pattern = X[-1]
    actual_future_label = y[-1]
    
    best_idx, sim_score, pred_label = predict_next_candle(current_pattern, history_X, history_y)
    
    direction_map = {0: "RIALZISTA / FLAT (BUY)", 1: "RIBASSISTA (SELL)"}
    
    print(f"--------------------------------------------------")
    print(f" Pattern Analizzato: Ultima candela M1 disponibile")
    print(f" Match Storico Identificato all'indice: {best_idx}")
    print(f" Grado di Similarità (Coseno): {sim_score:.4f}")
    print(f" Suggerimento Algoritmo: {direction_map[pred_label]}")
    print(f" Esito Reale del Mercato: {direction_map[actual_future_label]}")
    print(f"--------------------------------------------------")